In [ ]:
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_ROOT   = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "Aegis_Safe_Work"

In [ ]:
STAGE1_ROOT  = PROJECT_ROOT / "processed/stage1"
SPLITS_ROOT  = PROJECT_ROOT / "splits"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
MANIFEST_OUT = MANIFEST_DIR / "split_manifest.json"

In [ ]:
TRAIN_RATIO  = 0.9
SEED         = 42

In [ ]:
CLASS_MAP = {
    "fall":   "Fall",
    "normal": "Normal",
}


In [ ]:

def collect_stage1(label_dir: str) -> list[str]:
    """Return sorted list of .mp4 filenames in stage1/{label_dir}/."""
    src = STAGE1_ROOT / label_dir
    if not src.exists():
        raise FileNotFoundError(f"Stage1 source not found: {src}")
    files = sorted(p.name for p in src.iterdir() if p.suffix == ".mp4")
    return files


In [ ]:

def stratified_split(files: list[str], ratio: float, seed: int):
    """Shuffle and split a list into (train, val) by ratio."""
    rng = random.Random(seed)
    shuffled = files.copy()
    rng.shuffle(shuffled)
    n_train = int(len(shuffled) * ratio)   # floor via int()
    return shuffled[:n_train], shuffled[n_train:]



In [ ]:

def make_symlinks(filenames: list[str], src_dir: Path, dst_dir: Path) -> int:
    """
    Create symlinks in dst_dir pointing to absolute paths in src_dir.
    Returns count of symlinks created.
    """
    dst_dir.mkdir(parents=True, exist_ok=True)
    created = 0
    for fname in filenames:
        src  = src_dir / fname
        link = dst_dir / fname
        if link.exists() or link.is_symlink():
            link.unlink()              # remove stale symlink if re-running
        link.symlink_to(src.resolve())
        created += 1
    return created


In [ ]:
def verify_symlinks(dst_dir: Path) -> tuple[int, int]:
    """Return (valid, broken) symlink counts in dst_dir."""
    valid  = 0
    broken = 0
    for p in dst_dir.iterdir():
        if p.is_symlink():
            if p.exists():
                valid += 1
            else:
                broken += 1
    return valid, broken


In [ ]:
def main():
    import shutil # Import shutil for rmtree

    print("=" * 60)
    print("Aegis-Safe-Work | Stratified Split + Symlinks")
    print("=" * 60)

    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

    # Use a local path for symlinks to avoid "Operation not supported" error on Google Drive
    # The manifest will still be written to MANIFEST_OUT on Google Drive.
    LOCAL_SPLITS_ROOT = Path("/content/splits_local")
    if LOCAL_SPLITS_ROOT.exists():
        shutil.rmtree(LOCAL_SPLITS_ROOT) # Clean up previous run if it exists
    LOCAL_SPLITS_ROOT.mkdir(parents=True, exist_ok=True)


    manifest = {
        "metadata": {
            "created":     datetime.now(timezone.utc).isoformat(),
            "split_ratio": TRAIN_RATIO,
            "seed":        SEED,
            "classes":     list(CLASS_MAP.values())
        },
        "train": {},
        "val":   {}
    }

    total_train = 0
    total_val   = 0

    for label_dir, label_cap in CLASS_MAP.items():
        print(f"\n[{label_cap.upper()}]")

        # Collect
        files = collect_stage1(label_dir)
        print(f"  Found in stage1/{label_dir}/: {len(files)} videos")

        # Split
        train_files, val_files = stratified_split(files, TRAIN_RATIO, SEED)
        print(f"  Train: {len(train_files)} | Val: {len(val_files)}")

        # Source dir (absolute paths for symlink targets)
        src_dir = (STAGE1_ROOT / label_dir).resolve()

        # Create symlinks in the local directory
        train_dst = LOCAL_SPLITS_ROOT / "train" / label_cap
        val_dst   = LOCAL_SPLITS_ROOT / "val"   / label_cap

        n_train = make_symlinks(train_files, src_dir, train_dst)
        n_val   = make_symlinks(val_files,   src_dir, val_dst)

        print(f"  Symlinks created -> train: {n_train} | val: {n_val}")

        # Verify
        tv, tb = verify_symlinks(train_dst)
        vv, vb = verify_symlinks(val_dst)
        print(f"  Verify train: {tv} valid, {tb} broken")
        print(f"  Verify val  : {vv} valid, {vb} broken")

        # Manifest
        manifest["train"][label_cap] = train_files
        manifest["val"][label_cap]   = val_files

        total_train += len(train_files)
        total_val   += len(val_files)

    # Finalize metadata counts
    manifest["metadata"]["total_videos"] = total_train + total_val
    manifest["metadata"]["train_count"]  = total_train
    manifest["metadata"]["val_count"]    = total_val

    # Write manifest
    with open(MANIFEST_OUT, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"\n[MANIFEST] Written to: {MANIFEST_OUT}")

    # Final summary
    print("\n" + "=" * 60)
    print("SPLIT SUMMARY")
    print("=" * 60)
    for label_cap in CLASS_MAP.values():
        tr = len(manifest["train"][label_cap])
        vl = len(manifest["val"][label_cap])
        print(f"  {label_cap:<8} -> train: {tr:>5} | val: {vl:>4} | total: {tr+vl}")
    print(f"  {'TOTAL':<8} -> train: {total_train:>5} | val: {total_val:>4} | total: {total_train+total_val}")
    print("=" * 60)
    print("Split complete. Next: ETL Stage 2 reads from splits/ via manifest.")

In [ ]:
main()

Aegis-Safe-Work | Stratified Split + Symlinks

[FALL]
  Found in stage1/fall/: 902 videos
  Train: 811 | Val: 91
  Symlinks created -> train: 811 | val: 91
  Verify train: 811 valid, 0 broken
  Verify val  : 91 valid, 0 broken

[NORMAL]
  Found in stage1/normal/: 1183 videos
  Train: 1064 | Val: 119
  Symlinks created -> train: 1064 | val: 119
  Verify train: 1064 valid, 0 broken
  Verify val  : 119 valid, 0 broken

[MANIFEST] Written to: /content/drive/MyDrive/Aegis_Safe_Work/manifests/split_manifest.json

SPLIT SUMMARY
  Fall     -> train:   811 | val:   91 | total: 902
  Normal   -> train:  1064 | val:  119 | total: 1183
  TOTAL    -> train:  1875 | val:  210 | total: 2085
Split complete. Next: ETL Stage 2 reads from splits/ via manifest.


## Coonfirm directory exist

In [ ]:
!ls /content/drive/MyDrive/Aegis_Safe_Work/splits/train

Fall


In [ ]:

!ls /content/drive/MyDrive/Aegis_Safe_Work/splits/

train


In [ ]:
!ls /content/drive/MyDrive/Aegis_Safe_Work/splits/val/

ls: cannot access '/content/drive/MyDrive/Aegis_Safe_Work/splits/val/': No such file or directory


In [ ]:
!ls /content/drive/MyDrive/Aegis_Safe_Work/splits/val/Fall/ | wc -l
!ls /content/drive/MyDrive/Aegis_Safe_Work/splits/val/Normal/ | wc -l

ls: cannot access '/content/drive/MyDrive/Aegis_Safe_Work/splits/val/Fall/': No such file or directory
0
ls: cannot access '/content/drive/MyDrive/Aegis_Safe_Work/splits/val/Normal/': No such file or directory
0


In [ ]:
import json
with open("/content/drive/MyDrive/Aegis_Safe_Work/manifests/split_manifest.json") as f:
    m = json.load(f)
print("val fall  :", len(m["val"]["Fall"]))
print("val normal:", len(m["val"]["Normal"]))

val fall  : 91
val normal: 119


In [ ]:
import json
from pathlib import Path

The splits/val directories was never created,  it ought to be re created ASAP

In [ ]:
DRIVE_ROOT   = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "Aegis_Safe_Work"
STAGE1_ROOT  = PROJECT_ROOT / "processed/stage1"
SPLITS_ROOT  = PROJECT_ROOT / "splits"

In [ ]:
with open(PROJECT_ROOT / "manifests/split_manifest.json") as f:
    m = json.load(f)

In [ ]:

for label_cap, label_dir in [("Fall", "fall"), ("Normal", "normal")]:
    dst = SPLITS_ROOT / "val" / label_cap
    dst.mkdir(parents=True, exist_ok=True)
    src_dir = (STAGE1_ROOT / label_dir).resolve()
    for fname in m["val"][label_cap]:
        link = dst / fname
        if link.is_symlink():
            link.unlink()
        link.symlink_to(src_dir / fname)
    valid  = sum(1 for p in dst.iterdir() if p.is_symlink() and p.exists())
    broken = sum(1 for p in dst.iterdir() if p.is_symlink() and not p.exists())
    print(f"val/{label_cap}: {valid} valid, {broken} broken")

OSError: [Errno 95] Operation not supported: '/content/drive/MyDrive/Aegis_Safe_Work/processed/stage1/fall/Fall_0688.mp4' -> '/content/drive/MyDrive/Aegis_Safe_Work/splits/val/Fall/Fall_0688.mp4'

val fall  : 91
val normal: 119
train fall : 811
train normal: 1064

In [ ]:
# tensors/train/fall/Fall_0001.npy
# tensors/train/normal/Normal_0001.npy
# tensors/val/fall/Fall_0002.npy
# tensors/val/normal/Normal_0002.npy